# Lead Paint

Data from: https://www.schools.nyc.gov/school-life/space-and-facilities/space-and-facilities-reports/lead-based-paint

According to DOE, monitoring and remediation of lead paint in school buildings only "applies to all buildings serving kids under six built before 1985". This data only contains "results of the inspections for all classrooms serving students in first grade or under".

Standard response protocols for lead-based paint are only initiated when there is paint peeling already. See this description of protocols from the above described in the webpage from the above link:


If your child is under the age of six and their school was built before 1985:
- Their classroom is inspected and if there is any peeling paint, it is addressed
- If there is peeling paint, an independent contractor tests whether the paint is lead-based paint, and, if it is, a safety plan is developed.
- If the paint is lead-based paint or is paint of an unknown origin, the peeling paint is stripped and then sealed with a certified primer and painted over. Deteriorating paint on impact points, such as radiators and window sills is completely removed. This is abatement. If this work is occurring in a classroom, students are relocated in the interim. If it is occurring in a common space, it is made inaccessible to students so the work is safe.
- Once the lead-based paint has been abated, an independent certified inspector technician takes dust wipes that are tested to make sure the room is safe for children to reenter. This completes remediation.
- Custodian Engineers continue to visually inspect classrooms regularly.

In [ ]:
import pandas as pd

In [ ]:
# TODO: what is a cohort and why are there 3 a year?
# TODO: download data for last 5 years and add to the below

In [ ]:
cohort1_2025_fp = '../data/raw_data/DOE/Lead Paint/2025-cohort-1-lbp-report-final.xlsx'
cohort2_2025_fp = '../data/raw_data/DOE/Lead Paint/2025-cohort-2-lbp-report-final.xlsx'
cohort3_2025_fp = '../data/raw_data/DOE/Lead Paint/2025-cohort-3-lbp-report-final.xlsx'

c1_25_df = pd.read_excel(cohort1_2025_fp)
c2_25_df = pd.read_excel(cohort2_2025_fp)
c3_25_df = pd.read_excel(cohort3_2025_fp)

c1_25_df = c1_25_df.drop(columns=['Unnamed: 0', 'BOROUGH', 'GEOGRAPHICAL DISTRICT'])
c2_25_df = c2_25_df.drop(columns=['Unnamed: 0', 'BOROUGH', 'GEOGRAPHICAL DISTRICT'])
c3_25_df = c3_25_df.drop(columns=['BOROUGH', 'GEOGRAPHICAL DISTRICT'])

In [ ]:
all_cohorts_25 = pd.concat([c1_25_df, c2_25_df, c3_25_df])
all_cohorts_25['BUILDING CODE'] = all_cohorts_25['BUILDING CODE'].str.strip().str.upper()
all_cohorts_25['ROOM'] = all_cohorts_25['ROOM'].str.strip().str.upper()
# Check that building codes are all standardized
assert (all_cohorts_25['BUILDING CODE'].str.contains(r'^[XQMKR]\d{3}$')).all()

This data doesn't include any un-remediated classrooms, which means they're only reporting _after_ abatement is done. I personally think this is a bit underhanded, because it gives us no sense of how many schools are actively undergoing lead abatement or where lead abatement might not have occurred even after a long time after it was found. We have no way of calculating the average response time of abatement after initial reporting.

Given this, my curren thought is that the best we can do with this data is show the schools where the highest number of abatement cases has occurred within the last N years (thinking maybe last 5 years?). The intention here would be to show the schools with the most lead contamination. However, that's not actually what we're able to do with this data if they never update until abatement has already occurred. Instead, we get survivor bias, which may reflect the schools that are (1) most on top of their requirements to detect lead and/or (2) most effective in abating lead once it is found.

In addition, this data is very limited, since it only covers schools with students in first grade or younger. We have no sense of the lead exposure for students more than 6 years old.

I'd like to do some searching around online to see if there's something like the IBO report from an independent body that makes sense of this data vis-a-vis student health. Ideally, I'd like to find a way to approximate how much unabated lead paint is in all school buildings using this data or other datasets. It might be best to just flag schools built before 1965 or 1985 (DOE adds a 20-year buffer for the buildings it inspects) as likely to have lead paint.

Still need to find lead pipe data. I know there's data on this for regular buildings in NYC (saw a map once). Maybe that source or another similar one could be used to locate lead pipes in schools?

In [ ]:
# NOTE: it's suspicious to me that there are NO un-remediated classrooms. That suggests that in-progress work is not being reported, and it's only reported when it's complete.

# Not seeing any logically contradictory results, which is nice. If there's deteriorated paint and lead is in it, 'REMEDIATION REQUIRED' is always "YES"
all_cohorts_25[
    # (all_cohorts_25['DETERIORATED PAINT']=='NO')
    # &
    (all_cohorts_25['PRESENCE OF LBP IN DETERIORATED PAINT']=='YES')
    &
    (all_cohorts_25['REMEDIATION COMPLETE']!='YES')
    # &
    # (all_cohorts_25['REMEDIATION REQUIRED']!='YES')
    ]


In [ ]:
# Check that there are no rooms where more than 1 instance of LBP was found
# This allows us to calculate the proportion of rooms with LBP by just dividing unique rooms by instances of LBP
assert not (all_cohorts_25.groupby(['BUILDING CODE', 'ROOM'])['PRESENCE OF LBP IN DETERIORATED PAINT'].transform(lambda s: (s=='YES').sum()) > 1).any()

In [ ]:
lbp_per_school_bldg = all_cohorts_25.groupby('BUILDING CODE').agg(
        {
            'ROOM': 'nunique', 
            'PRESENCE OF LBP IN DETERIORATED PAINT': lambda s: (s=='YES').sum()
        }
    ).rename(columns={'ROOM': 'rooms', 'PRESENCE OF LBP IN DETERIORATED PAINT': 'lbp_found'})

In [ ]:
lbp_per_school_bldg[lbp_per_school_bldg['lbp_found']>0].sort_values('lbp_found', ascending=False)